# Notebook 03 -- strata and split

**Asks:** draw the model-free sample and freeze the SEED/TEST split.

**Reads:** `openends_long.csv` (notebook 00), `facet_dictionary_scores.csv`
(notebook 01), and `retribution_scored.csv` (Track 1's R Markdown writes it and
notebook 00 carries it into the folder).

**Writes:** `strata_sample.csv`, `gold_split_assignment.csv`.

This notebook draws the roughly 250-300 response sample the study uses as its
anchor, and partitions that sample ONCE into a SEED half and a TEST half. Both
jobs are done without looking at any model output, and neither needs a human
coder: the sample comes from a random route, a survey route and an open
read-through route, and the split is frozen from `RANDOM_SEED`. The optional
human-coding arm reads `strata_sample.csv` later without anything here being
re-run.

**`retribution_scored.csv` is marked optional, and it is not a minor input.**
Optional means only that its absence cannot stop the notebook. What it actually
decides is the stratifier for stratum (a) and what stratum (b) IS: without it,
stratum (b) becomes the logical inverse of the registered one -- the responses
the word list DID flag, instead of the ones it missed -- and about 95% of the
rows of the frozen split change. Run this notebook with that file in the folder.
When it is absent the notebook says so in capitals, three times, and records the
route it took in the manifest entry for both files it writes.

In [1]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "03"                     # this notebook's number
NB_NAME     = "strata_and_split"       # this notebook's job, in two words
NB_ASKS     = "Draw the model-free sample and freeze the SEED/TEST split."

## Setup

In [2]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

NOTEBOOK 03 -- strata_and_split
  Draw the model-free sample and freeze the SEED/TEST split.
  mode: RUN_MODE=full, HUMAN_GOLD=False
  folder: /content/retribution_nlp
  this folder is empty: notebook 00 is the place to start.


## Load

In [3]:
# === LOAD: every input through need() ===
# WHAT THIS CELL DOES: fetches the three inputs and leaves them in df, dict_df and
# survey. Two are required; the third is optional and the notebook prints, three
# times over, which route it took without it.
# CONSUMES:
#   openends_long.csv (notebook 00) -- one row per response_id x item_type, which the
#     next cell checks rather than assumes, carrying the verbatim text. The text has to travel into the drawn sample, because the
#     text is the only thing a reader of that sample reads.
#   facet_dictionary_scores.csv (notebook 01) -- the word-list stream. It supplies
#     the facet counts that stratum (b) uses as its "the word list found nothing
#     here" signal.
#   retribution_scored.csv -- Track 1's scored survey file, N = 948 after the
#     preregistered exclusions of prereg 6.2. It is written by the R Markdown rather
#     than by this chain; notebook 00 carries it into the folder and registers it,
#     so it is usually already here. OPTIONAL only in the sense that its absence
#     cannot stop the run; see the route banner below for what its absence costs.
# PRODUCES: df, dict_df, survey (or None). Writes nothing.

import pandas as pd
import numpy as np

OPENENDS_PATH = need("openends_long.csv", produced_by="00")
DICT_PATH     = need("facet_dictionary_scores.csv", produced_by="01")
SURVEY_PATH   = need("retribution_scored.csv", produced_by="00", optional=True)

# keep_default_na=False so a participant who literally answered "N/A" keeps that text
# instead of arriving as a missing value. The draw is over non-blank responses, and a
# real answer read as missing would drop out of the pool with nobody noticing.
df = pd.read_csv(OPENENDS_PATH, keep_default_na=False)
df["text"] = df["text"].fillna("").astype(str)
dict_df = pd.read_csv(DICT_PATH)
survey  = pd.read_csv(SURVEY_PATH) if SURVEY_PATH else None

# WHAT NOTEBOOK 00 DID TO THE CORPUS, read out of the manifest rather than guessed at
# here. Notebook 00 cell 15 is where the preregistered 6.2 exclusions are applied
# (1015 -> 965 -> 958 -> 948) and its done() call records "applied" or "not run" in the
# note. This notebook has no way of re-deriving that, and printing it is the difference
# between a reader knowing which corpus is in front of them and assuming.
NB00_NOTE = _load_manifest()["files"].get("openends_long.csv", {}).get("note", "")
print(f"  notebook 00 recorded: {NB00_NOTE or '(nothing -- file not registered)'}")
print(f"  long open-ends: {len(df)} rows. Dictionary scores: {len(dict_df)} rows.")

  [ok]   openends_long.csv  (written unknown time)
  [ok]   facet_dictionary_scores.csv  (written unknown time)
  [ok]   retribution_scored.csv  (written unknown time)
  notebook 00 recorded: (nothing -- file not registered)
  long open-ends: 2844 rows. Dictionary scores: 2844 rows.


In [4]:
# === THE CONTRACT ON THE TWO REQUIRED INPUTS, CHECKED BEFORE ANY OF IT IS USED ===
# WHAT THIS CELL DOES: proves the two required files are the shape everything below
# assumes. It computes no result. It exists so that a broken input says so here, by
# name, instead of surfacing four cells down as an AttributeError or -- far worse --
# as a number that looks fine.
# CONSUMES: df, dict_df.   PRODUCES: nothing.
#
# COLUMNS. issubset rather than equality, so a file carrying extra columns still
# passes. These six are the columns THIS notebook reads: n_words builds the pool,
# desert stratifies stratum (a), revenge and suffering define stratum (b), and the id
# pair keys every merge. Without the check, a renamed column arrives as
# "AttributeError: 'DataFrame' object has no attribute 'n_words'", which names the
# symptom and not the file that caused it.
_need_dict = {"response_id", "item_type", "n_words", "desert", "revenge", "suffering"}
assert _need_dict.issubset(set(dict_df.columns)), (
    "facet_dictionary_scores.csv (notebook 01) is missing the columns this notebook "
    f"reads: {sorted(_need_dict - set(dict_df.columns))}")
_need_long = {"response_id", "item_type", "text"}
assert _need_long.issubset(set(df.columns)), (
    "openends_long.csv (notebook 00) is missing the columns this notebook reads: "
    f"{sorted(_need_long - set(df.columns))}")

# ONE ROW PER PARTICIPANT PER ITEM. ENFORCED HERE, for these two files, by these two
# lines. The whole chain is keyed on the (response_id, item_type) pair -- 948
# participants x 3 open-ended items = 2,844 rows -- and nothing outside this cell
# guarantees the files still hold that shape by the time they reach this notebook. A
# file can be re-exported, hand-edited in a spreadsheet, or concatenated with itself.
# What a duplicated row would do here: the text merge below would fan the pool out, one
# participant could be drawn into the sample twice, and every count this notebook prints
# would be quietly wrong while still looking entirely reasonable.
_dup_long = int(df.duplicated(["response_id", "item_type"]).sum())
assert _dup_long == 0, (
    f"openends_long.csv carries {_dup_long} duplicate (response_id, item_type) rows. "
    "It must hold exactly one row per participant per item. Re-run notebook 00.")
_dup_dict = int(dict_df.duplicated(["response_id", "item_type"]).sum())
assert _dup_dict == 0, (
    f"facet_dictionary_scores.csv carries {_dup_dict} duplicate (response_id, "
    "item_type) rows. Re-run notebook 01.")
print(f"  one row per participant per item, checked in both required files: "
      f"{len(df)} rows / {df.response_id.nunique()} participants (open-ends), "
      f"{len(dict_df)} rows / {dict_df.response_id.nunique()} (word list).")

  one row per participant per item, checked in both required files: 2844 rows / 948 participants (open-ends), 2844 rows / 948 (word list).


In [5]:
# === THE SURVEY FILE: CHECKED, AND NEVER ALLOWED TO STOP THE RUN ===
# WHAT THIS CELL DOES: decides whether retribution_scored.csv can be used, and says so.
# CONSUMES: survey.   PRODUCES: survey (possibly set back to None), SURVEY_ROUTE.
#
# WHY NOTHING HERE RAISES. This file is optional, and a defect in an optional input
# must never kill the analyses that do not use it. So every check below prints what is
# wrong and sets SURVEY_ROUTE to "not run", and the notebook carries on down the
# fallback route. SURVEY_ROUTE is the single flag the tertile cell, the stratum (b)
# cell, the banner and both done() notes read, so there is one decision and one record
# of it rather than the same condition re-derived in five places.
SURVEY_ROUTE = "not run"      # "used" once the file has passed the checks below
if survey is not None:
    # The participant id column in the survey export is ResponseId. Renamed rather than
    # merged on two names, so the join key is one column everywhere.
    if "ResponseId" in survey.columns and "response_id" not in survey.columns:
        survey = survey.rename(columns={"ResponseId": "response_id"})
    _dup_survey = (int(survey["response_id"].duplicated().sum())
                   if "response_id" in survey.columns else 0)
    if "response_id" not in survey.columns:
        print("  [not run] retribution_scored.csv has no ResponseId / response_id "
              "column, so it cannot be joined to the corpus.")
        print(f"            Columns found: {list(survey.columns)[:8]} ...")
        survey = None
    elif _dup_survey:
        # NOT silently de-duplicated. This file is supposed to be one row per
        # participant; if it is not, then whichever of a participant's rows happened to
        # come first would decide their `deserved` tertile and their stratum (b) rank,
        # and the frozen split would rest on a coin toss nobody threw.
        print(f"  [not run] retribution_scored.csv has {_dup_survey} duplicate "
              "response_id rows, so it is not one row per participant and cannot be")
        print("            trusted to say who the analytic sample is. Re-export it "
              "from 01_main_analysis.Rmd.")
        survey = None
    else:
        SURVEY_ROUTE = "used"
        print(f"  survey scaffold: {len(survey)} participants, one row each.")

# RUN_MODE = "smoke" trims to a handful of PARTICIPANTS, never to a handful of rows: a
# row trim would cut a person in half and every three-rows-per-person check in the chain
# would fail for a reason that has nothing to do with the data. The stratum budgets
# below are larger than a smoke pool, and every draw is guarded by min() or head(), so a
# smoke run simply takes whatever is there. A smoke split is NOT the frozen split.
if RUN_MODE == "smoke":
    _keep = list(pd.unique(df["response_id"]))[:20]
    df = df[df["response_id"].isin(_keep)].copy()
    dict_df = dict_df[dict_df["response_id"].isin(_keep)].copy()
    print(f"  SMOKE MODE: trimmed to {len(_keep)} participants, {len(df)} rows.")

  survey scaffold: 948 participants, one row each.


## Do

In [6]:
# === WHY THIS SAMPLE IS DRAWN WITHOUT USING ANY MODEL OUTPUT ===
# The single design decision this notebook exists to protect.
# CONSUMES: nothing.   PRODUCES: FACET_NAMES and the three stratum budgets.
#
# The sample drawn here is the validity head: it is the set of responses every other
# stream is checked against, and (where the optional human arm is switched on) the
# set a human reads. If the hard strata were seeded by the very models the sample
# validates, only cases the models already find legible would ever be drawn in; a
# blind-spot case would never enter the sample; and a check against that sample could
# never detect the blind spot it was built to catch. Agreement on a model-curated
# sample is accuracy on the cases the model already found, not accuracy on the corpus
# (MEASUREMENT_DESIGN.md, design 8). Hence three model-free routes:
#   (a) a stratified RANDOM slice -- gives every response, blind spots included, a
#       nonzero chance of being drawn;
#   (b) a SURVEY-signal disagreement stratum -- high self-reported revenge/suffering
#       whose open-end the dictionary scores 0, which is exactly where a shared blind
#       spot lives, and which uses only closed-ended survey data as the flag;
#   (c) an OPEN read-through reserve with no filter at all, to surface vengeful
#       expressions no model and no dictionary tagged.
# HAZARD: never add a fourth stratum seeded by ensemble or embedding output. It would
# void the guarantee by construction, and it would do so silently.

# The nine facets, in the order every file in the chain lists them. Written out here
# rather than imported so this notebook stands on its own; the blank coder columns at
# the bottom follow the same order as the scored files.
FACET_NAMES = ["desert", "proportionality", "censure", "moral_balance",
               "revenge", "suffering",
               "deterrence", "incapacitation", "rehabilitation"]

# The budget, split across the three strata. Small, targeted and one-time by design
# (design 8): an anchor and a calibrator, not a parallel full-corpus coding effort.
STRATUM_B_N    = 60     # survey-model disagreement
STRATUM_A_CELL = 22     # per deserved_tertile x item_type cell, 9 cells -> up to 198
STRATUM_C_N    = 50     # open read-through reserve
print(f"Budget: (b) {STRATUM_B_N} + (a) up to 9 x {STRATUM_A_CELL} + (c) "
      f"{STRATUM_C_N}, a target of roughly 250-300 rows.")

Budget: (b) 60 + (a) up to 9 x 22 + (c) 50, a target of roughly 250-300 rows.


In [7]:
# === THE POOL: non-blank responses, with their text attached ===
# CONSUMES: dict_df, df.   PRODUCES: gold_pool -- one row per non-blank response, with
#           the word-list counts and the verbatim text side by side.
#
# ONE ROW PER PARTICIPANT, the rule that governs all three strata. A reader taking three
# items from the same participant is not reading them independently -- the second and
# third are read in the light of the first -- so agreement across them would be earned
# by the participant rather than by the codebook, and any agreement statistic computed
# on them would be inflated. The three strata are therefore drawn in SEQUENCE against a
# shared `used` set of response_ids, scarcest first: (b) can only be filled from a thin
# targeted region, (a) draws from the whole pool, (c) takes what is left. One
# participant contributes exactly one row, and the assert in the assembly cell below
# ("A participant appears on more than one row") is what enforces it.

# Blank open-ends are excluded because there is nothing there to read: a blank row
# would consume budget and contribute nothing to any agreement or gate number.
gold_pool = dict_df[dict_df.n_words > 0].copy()

# facet_dictionary_scores.csv carries ids, counts and rates but deliberately not the
# response itself, so the verbatim text is merged back from the long file on the
# (response_id, item_type) key the whole chain uses.
# THE TWO THINGS THAT CAN GO WRONG IN THIS MERGE, and the guard for each:
#   a right-hand frame holding two rows for one key FANS THE POOL OUT, so one
#   participant can be drawn into the sample twice and every count printed below is
#   wrong while still looking reasonable -- caught by the row count, which cannot
#   change on a left join unless the right-hand side has multiplied a key;
#   keys that do not line up leave the text MISSING, which would send blank rows to a
#   reader -- caught by the notna check.
# The count is checked HERE and not only at the load cell above, because notebook cells
# are re-run one at a time: the load cell proved openends_long.csv was one row per
# participant per item at the moment IT ran, which says nothing about the `df` that is
# in memory now.
_n_pool = len(gold_pool)
gold_pool = gold_pool.merge(df[["response_id", "item_type", "text"]],
                            on=["response_id", "item_type"], how="left")
assert len(gold_pool) == _n_pool, (
    f"The text merge changed the pool from {_n_pool} to {len(gold_pool)} rows. `df` is "
    "not one row per (response_id, item_type) any more, so the merge multiplied rows "
    "and a participant could be drawn into the sample twice.")
assert gold_pool["text"].notna().all(), \
    "A pooled row lost its text in the merge -- check the response_id / item_type keys."
gold_pool["text"] = gold_pool["text"].astype(str)
print(f"Pool: {len(gold_pool)} non-blank responses from "
      f"{gold_pool.response_id.nunique()} participants.")

Pool: 2837 non-blank responses from 948 participants.


In [8]:
# === IS THE POOL THE ANALYTIC SAMPLE? THIS CELL CHECKS; NOTEBOOK 00 DECIDES ===
# CONSUMES: gold_pool, survey, SURVEY_ROUTE.
# PRODUCES: gold_pool (restricted only if notebook 00 did not restrict it), SURVEY_ROUTE.
#
# WHO OWNS THE RESTRICTION. Notebook 00, in its cell 15, applies prereg 6.2 (Finished,
# Progress, the two attention checks: 1015 -> 965 -> 958 -> 948) when it writes
# openends_long.csv, and records "applied" or "not run" in the manifest note printed by
# the load cell above. On a correctly built chain the filter below therefore removes
# NOTHING and this cell is a verification, which is what its output says. A non-zero
# drop means notebook 00 ran without the survey file in the folder.
if SURVEY_ROUTE == "used":
    _analytic = set(survey["response_id"].astype(str))
    _in = gold_pool["response_id"].astype(str).isin(_analytic)
    _matched = gold_pool.loc[_in, "response_id"].nunique()
    # TRIPWIRE, the same one notebook 00 cell 15 carries. retribution_scored.csv comes
    # from the R Markdown, so nothing guarantees its id column lines up with the
    # Qualtrics export -- a hand-edited copy or a re-exported scaffold need not. If it
    # does not, the filter throws away every row, and the failure surfaces three cells
    # later out of pd.qcut as "Bin labels must be one fewer than the number of bin
    # edges", which points at the tertile and not at the ids. Caught here instead, and
    # the survey route is dropped rather than the notebook: an optional file may cost
    # the analyses that use it, never the ones that do not.
    if _matched == 0:
        print("  !! THE ANALYTIC RESTRICTION MATCHED NOTHING: not one participant in")
        print("     the corpus appears in retribution_scored.csv. That means the two id")
        print("     columns do not line up, not that everyone was excluded. Check that")
        print("     its ResponseId values look like the response_id values in the")
        print("     Qualtrics export.")
        print("  [not run] The survey route is dropped. The pool is left whole and the")
        print("            fallback strata below are used instead.")
        SURVEY_ROUTE, survey = "not run", None
    else:
        if _matched < 0.5 * len(_analytic):
            print(f"  !! WARNING: only {_matched} of the {len(_analytic)} participants "
                  "in retribution_scored.csv matched the corpus. A near-complete match")
            print("     is expected. A shortfall this large means the two id columns")
            print("     disagree, and the sample drawn below -- the file that goes to")
            print("     OSF -- would be drawn from that fraction of the corpus only.")
        _before, _parts_before = len(gold_pool), gold_pool.response_id.nunique()
        gold_pool = gold_pool[_in.values].copy()
        if len(gold_pool) == _before:
            print(f"  Analytic sample verified: all {_matched} pool participants are in "
                  "retribution_scored.csv, so nothing was dropped here.")
        else:
            print(f"  Pool restricted HERE: {_before} -> {len(gold_pool)} rows, "
                  f"{_parts_before - _matched} participants removed.")
            print("  Notebook 00 owns this step (its cell 15); a non-zero number means it")
            print("  ran without retribution_scored.csv and could not apply prereg 6.2.")
            print("  The rest of the chain still scores the unrestricted corpus. Put the")
            print("  file beside the raw export and re-run notebook 00.")
else:
    print("  [not run] Analytic-sample check: retribution_scored.csv is not usable here,")
    print("            so which participants prereg 6.2 retains is not knowable in this")
    print("            notebook. Whether the corpus is already restricted is recorded in")
    print("            notebook 00's manifest note, printed by the load cell above.")

  Analytic sample verified: all 948 pool participants are in retribution_scored.csv, so nothing was dropped here.


In [9]:
# === ATTACH THE SURVEY COMPOSITES TO THE POOL ===
# CONSUMES: gold_pool, survey, SURVEY_ROUTE.   PRODUCES: gold_pool + survey_ columns.
#
# The composites are prefixed with survey_ so they do NOT collide with the
# identically-named dictionary facet columns (revenge, suffering) after the merge --
# a silent bug otherwise. Without the prefix, pandas would emit revenge_x / revenge_y
# and every reference to `revenge` below would either raise or, worse, quietly pick up
# the survey scale where the dictionary count was meant. The disagreement stratum
# compares exactly those two, so the collision would land precisely where it does the
# most damage.
SURVEY_PREFIX = "survey_"
if SURVEY_ROUTE == "used":
    src = [c for c in ["deserved", "revenge", "suffering"] if c in survey.columns]
    keep = survey[["response_id"] + src].rename(
        columns={c: SURVEY_PREFIX + c for c in src})
    _n_pool = len(gold_pool)
    # how="left" so the merge itself cannot drop a pool row: a participant missing from
    # the survey file keeps their row and gets NaN composites, and the strata that need
    # a composite skip them by hand rather than by silent disappearance. A left join
    # can still GROW the pool, and only one way: the right-hand frame holding two rows
    # for one participant. The load cell refused such a file, so this count is the
    # second line of defence, for the case where `survey` was rebuilt in memory after
    # that cell ran -- which is what happens when cells are executed out of order.
    gold_pool = gold_pool.merge(keep, on="response_id", how="left")
    assert len(gold_pool) == _n_pool, (
        f"The survey merge changed the pool from {_n_pool} to {len(gold_pool)} rows. "
        "`survey` is not one row per response_id any more, so a participant would be "
        "drawn into the sample twice.")
    print(f"  Attached {len(src)} survey composites: "
          f"{', '.join(SURVEY_PREFIX + c for c in src)}.")
else:
    print("  No survey composites attached.")

  Attached 3 survey composites: survey_deserved, survey_revenge, survey_suffering.


In [10]:
# === THE deserved TERTILE THAT STRATUM (a) IS DRAWN WITHIN ===
# CONSUMES: gold_pool, SURVEY_ROUTE.
# PRODUCES: gold_pool["deserved_tertile"], STRATUM_A_BY, STRATUM_A_SOURCE.
#
# Stratifying on the survey retribution composite is what makes stratum (a) span the
# full range of retributive attitude rather than over-representing the middle
# (design 8: deciles or tertiles of `deserved`).
if (SURVEY_PREFIX + "deserved") in gold_pool.columns \
        and gold_pool[SURVEY_PREFIX + "deserved"].notna().any():
    # qcut, which cuts a column into equal-sized bins, is applied to the RANK and not
    # to the raw composite. `deserved` is a mean of four 7-point items, so it has
    # heavy ties; qcut on the raw values would hit duplicate bin edges and fail.
    # rank(method="first") breaks ties by row order, which gives qcut three strictly
    # increasing edges and three near-equal cells.
    gold_pool["deserved_tertile"] = pd.qcut(
        gold_pool[SURVEY_PREFIX + "deserved"].rank(method="first"),
        3, labels=["low", "mid", "high"])
    STRATUM_A_SOURCE = "survey `deserved` composite"
else:
    # FALLBACK, and a weak one. The dictionary desert count is 0 on about 80% of the
    # corpus (2,267 of the 2,837 non-blank responses; 539 ones, 30 twos, one three), so
    # ranking it and cutting the rank into thirds does not produce three levels of
    # retributive attitude: it produces a low third and a middle third that are BOTH
    # entirely zeros and differ only by the order the rows sit in the CSV, plus a top
    # third that carries every non-zero count. Stratum (a) still draws from the whole
    # pool and is still model-free, which is the property that has to hold, but on this
    # route it is a random slice stratified on item_type and on nothing else. The
    # printed line is the record that this is what happened.
    # rank(method="first") makes every value distinct, so the bin edges cannot collide
    # and qcut always returns exactly three bins: no duplicates= argument is needed or
    # would ever be reached.
    gold_pool["deserved_tertile"] = pd.qcut(gold_pool["desert"].rank(method="first"),
                                            3, labels=["low", "mid", "high"])
    STRATUM_A_SOURCE = "dictionary desert count (no usable survey file)"
STRATUM_A_BY = ["deserved_tertile", "item_type"]
print(f"Stratum (a) tertiles come from the {STRATUM_A_SOURCE}.")

Stratum (a) tertiles come from the survey `deserved` composite.


In [11]:
# === (b) SURVEY-MODEL DISAGREEMENT: DRAWN FIRST, BECAUSE IT IS THE SCARCEST ===
# CONSUMES: gold_pool, SURVEY_ROUTE.   PRODUCES: used (empty), _pairs, STRATUM_B_ROUTE.
#
# High survey revenge or suffering, but a dictionary facet of 0.
# DRAWN FIRST because (a) and (c) can be filled from anywhere in the pool and this one
# only from a thin disagreement region, so it gets first call on the participants.
#
# This is the stratum that earns its keep. A participant who says on the closed-ended
# scales that they want revenge, and whose open-end the word list reads as containing
# none, is either expressing it in language no lexicon caught or not expressing it at
# all -- and only a reader of the text can say which. The flag uses survey data only,
# so nothing about it depends on the ensemble or on the embeddings.
#
# HOW THE REGION IS SELECTED, and why it is a RANKING rather than a threshold. The
# dictionary fires revenge on well under 1% of responses and suffering on about 2.5%,
# so "dictionary facet == 0" is true of almost the whole pool and removes almost
# nothing. Pairing it with a top-quartile cut still leaves a four-figure candidate
# set, and taking 60 at random out of that is a random draw wearing a targeted
# stratum's name -- most of it would sit just inside the quartile boundary, nowhere
# near the disagreement the stratum exists to sample. So the rows the dictionary
# scored 0 are RANKED by the survey scale and the top of that ranking is taken: the
# drawn rows are the strongest self-reported revenge/suffering in the corpus whose
# open-ends the word list read as carrying none, which is what "survey-model
# disagreement" is supposed to mean. Ties on the survey scale (the composites are
# means of four 7-point items, so ties are common at the ceiling) are broken by a
# RANDOM_SEED-seeded shuffle, so the draw is targeted and still reproducible.

# The running set of participants already drawn. Every stratum filters against it and
# then adds to it, which is what makes the one-row-per-participant rule hold across
# the whole sample instead of only inside a single stratum.
used = set()

# A pair is (survey column, dictionary column) for the same facet. Both must be
# present for the comparison to mean anything.
_pairs = [(SURVEY_PREFIX + s, s) for s in ["revenge", "suffering"]
          if (SURVEY_PREFIX + s) in gold_pool.columns and s in gold_pool.columns]

# ONE decision, recorded once and read three times below: by the branch that builds the
# stratum, by the label written into the file, and by the banner. Deriving the condition
# again at each of those points is how a route ends up labelled as the other one.
STRATUM_B_ROUTE = "survey_disagreement" if (SURVEY_ROUTE == "used" and _pairs) \
                  else "dict_vengeful"

In [12]:
# === (b) CONTINUED: rank inside the region, interleave the two scales ===
# CONSUMES: gold_pool, _pairs, STRATUM_B_ROUTE.   PRODUCES: stratum_b, used.
if STRATUM_B_ROUTE == "survey_disagreement":
    _ranked = []
    for _k, (scale, fac) in enumerate(_pairs):
        cand = gold_pool[(gold_pool[fac] == 0) & gold_pool[scale].notna()].copy()
        # Shuffle first, then a STABLE sort on the survey score: rows tied at the top
        # of the scale keep the shuffled order, so the tie-break is random and seeded
        # rather than whatever order the export happened to arrive in.
        cand = cand.sample(frac=1.0, random_state=RANDOM_SEED)
        cand = cand.sort_values(scale, ascending=False, kind="mergesort")
        # One row per participant within this scale's ranking; the participant's
        # highest-ranked row is the one kept.
        cand = cand[~cand["response_id"].isin(used)].drop_duplicates(subset=["response_id"])
        cand["_rank"] = np.arange(len(cand))    # 0 = the strongest disagreement here
        cand["_scale_order"] = _k               # interleave key
        cand["disagree_scale"] = scale          # which scale put this row in the stratum
        _ranked.append(cand.head(STRATUM_B_N))
    # Interleave the two rankings -- revenge #1, suffering #1, revenge #2, and so on --
    # so the budget is split evenly between the two facets by construction and neither
    # can crowd the other out. A participant who tops both rankings is kept once, at
    # their better rank, and the next name down moves up to fill the place.
    stratum_b = (pd.concat(_ranked)
                   .sort_values(["_rank", "_scale_order"])
                   .drop_duplicates(subset=["response_id"])
                   .head(STRATUM_B_N)
                   .drop(columns=["_rank", "_scale_order"])
                   .copy())
else:
    # THE FALLBACK IS THE INVERSE OF THE REGISTERED STRATUM, not a weaker version of it.
    # It collects the rows the dictionary DID flag (any revenge or suffering hit) rather
    # than the ones it missed, because without survey data there is no independent
    # signal of a miss. Design 8 item (b) asks for the disagreement region; this is the
    # agreement region. It is still model-independent -- no ensemble, no embedding, so
    # the guarantee the sample exists to provide still holds -- but the stratum that
    # "earns its keep" is not in the sample at all, and the banner below says so.
    stratum_b = gold_pool[(gold_pool["revenge"] > 0) | (gold_pool["suffering"] > 0)].copy()
    stratum_b = (stratum_b.sample(frac=1.0, random_state=RANDOM_SEED)
                          .drop_duplicates(subset=["response_id"])
                          .head(STRATUM_B_N)
                          .copy())
# The label the file carries, taken from the route decided above rather than derived
# again here. It travels into gold_split_assignment.csv, so which route ran is legible
# from the artifact itself and not only from this run log.
stratum_b["stratum"] = STRATUM_B_ROUTE
used.update(stratum_b["response_id"])

In [13]:
# === (b) CHECK: what the drawn region looks like against the pool it came from ===
# CONSUMES: stratum_b, gold_pool, _pairs, STRATUM_B_ROUTE.   PRODUCES: nothing.
#
# Reported per scale, on the rows THAT scale selected. A drawn minimum sitting at or
# above the pool's upper quartile is the check that the ranking bit: these really are
# the strongest self-reported revenge/suffering scorers whose open-ends the word list
# read as carrying none. A stratum of 0 rows means its condition never fired -- most
# likely no survey file, or a dictionary facet that is empty -- and that is worth
# chasing down before the sample is used for anything.
if STRATUM_B_ROUTE == "survey_disagreement":
    for _scale, _fac in _pairs:
        _sel = stratum_b[stratum_b["disagree_scale"] == _scale]
        if len(_sel):
            print(f"  (b) {_scale}: {len(_sel)} rows, drawn range "
                  f"{_sel[_scale].min():.2f}-{_sel[_scale].max():.2f} "
                  f"(pool median {gold_pool[_scale].median():.2f}, "
                  f"p75 {gold_pool[_scale].quantile(0.75):.2f}, dictionary "
                  f"{_fac} == 0 on every drawn row)")
else:
    # The second of the three places the fallback is announced. It is a banner and not
    # a one-liner because of what it costs: run both ways on the same corpus, the two
    # gold_split_assignment.csv files share 17 of 308 rows.
    print("  " + "!" * 68)
    print("  !! THE REGISTERED STRATUM (b) WAS NOT DRAWN.")
    print("  !! Without a usable retribution_scored.csv there is no survey signal, so")
    print("  !! this run substitutes its INVERSE: the responses the word list DID flag")
    print("  !! as vengeful, instead of the high self-reported scorers it missed. The")
    print(f"  !! stratum is labelled {STRATUM_B_ROUTE} in both files, and stratum (a)")
    print("  !! loses its `deserved` stratifier at the same time. About 95% of the rows")
    print("  !! of the frozen split differ from a run with the file present.")
    print("  !! Put retribution_scored.csv in the working folder and re-run.")
    print("  " + "!" * 68)
    print(f"  (b) dictionary-vengeful fallback: {len(stratum_b)} rows drawn.")

  (b) survey_revenge: 29 rows, drawn range 6.75-7.00 (pool median 4.25, p75 5.00, dictionary revenge == 0 on every drawn row)
  (b) survey_suffering: 31 rows, drawn range 6.50-7.00 (pool median 3.50, p75 5.00, dictionary suffering == 0 on every drawn row)


In [14]:
# === (a) THE STRATIFIED RANDOM SLICE ===
# CONSUMES: gold_pool, STRATUM_A_BY, used.   PRODUCES: stratum_a, used.
def random_stratified(pool, by, n_per_cell, used):
    """Draw n_per_cell rows per stratum cell, reproducibly, one row per participant.

    `used` is read AND updated: a participant taken by an earlier cell (or by an
    earlier stratum) is out of the frame for every cell that follows.
    """
    picks = []
    # observed=True because deserved_tertile is a categorical: without it, pandas
    # hands back a cell for every label the category knows about, including ones no
    # row landed in. The empty ones are skipped a line below either way, but pandas
    # is changing this default, and pinning it keeps the nine cells the same nine
    # cells whichever version of pandas Colab is running that month.
    for _, g in pool.groupby(by, observed=True):
        g = g[~g["response_id"].isin(used)]
        if not len(g):
            continue
        # min() so a thin cell contributes everything it has instead of raising.
        take = min(n_per_cell, len(g))
        # The same RANDOM_SEED is passed to every cell: the draw is reproducible, and
        # the cells are not independently seeded.
        sel = g.sample(n=take, random_state=RANDOM_SEED)
        used.update(sel["response_id"])
        picks.append(sel)
    return pd.concat(picks) if picks else pool.iloc[0:0]


# deserved_tertile (3) x item_type (3) = 9 cells, 9 x 22 = up to 198 rows. item_type
# is a stratifier in its own right because register differs sharply across the three
# items (design 8b), so a draw that ignored it would under-sample the short forward
# item. Because a participant can be taken by only one cell, the nine cells are filled
# in groupby order and each draws from whoever is still free -- every response in the
# pool still has a nonzero chance of being drawn, which is the property this stratum
# exists to provide.
stratum_a = random_stratified(gold_pool, STRATUM_A_BY, STRATUM_A_CELL, used)
stratum_a["stratum"] = "random_stratified"
print(f"  (a) stratified random slice: {len(stratum_a)} rows.")

  (a) stratified random slice: 198 rows.


In [15]:
# === (c) THE OPEN READ-THROUGH RESERVE, AND THE ASSEMBLED SAMPLE ===
# CONSUMES: gold_pool, used, stratum_a, stratum_b.   PRODUCES: stratum_c, strata.
#
# About 50 fully random responses, no filter at all. No filter IS the point: (a) is
# stratified and (b) is targeted, so both carry an assumption about where the
# interesting cases are. This stratum carries none, and it is the route by which a
# vengeful phrasing that no model and no dictionary tagged can enter the sample at all.
# WHAT DESIGN 8 ASKS FOR NEXT, AND WHICH NOTEBOOK DOES IT: design 8 item (c) says the
# verbatim patterns found on this read-through are folded back into the word list. No
# notebook in this chain does that, and none can -- notebook 01 builds the word list
# and runs two notebooks before this one. Folding a pattern back is a hand edit to the
# facet patterns in notebook 01 followed by a re-run of 01 and 02, and it draws from the
# SEED half only (see the split cell). Nothing downstream is waiting on it.
open_pool = gold_pool[~gold_pool["response_id"].isin(used)]
stratum_c = (open_pool.sample(frac=1.0, random_state=RANDOM_SEED)
                      .drop_duplicates(subset=["response_id"])
                      .head(STRATUM_C_N)
                      .copy())
stratum_c["stratum"] = "open_readthrough"
used.update(stratum_c["response_id"])
print(f"  (c) open read-through reserve: {len(stratum_c)} rows.")

strata = pd.concat([stratum_a, stratum_b, stratum_c], ignore_index=True)

# The independence rule, enforced rather than described. Each stratum drew against
# `used`, so there is nothing here to dedupe; if this ever fires, a stratum stopped
# filtering and the sample would be handing a reader repeat participants.
assert strata["response_id"].nunique() == len(strata), (
    "A participant appears on more than one row -- the one-row-per-participant "
    "rule broke somewhere in the sequential draw.")

  (c) open read-through reserve: 50 rows.


In [16]:
# === THE PREREGISTERED TRIPWIRE: NO STRATUM DEFINED BY MODEL OUTPUT ===
# CONSUMES: strata, gold_pool.   PRODUCES: nothing.
#
# A tripwire on the stratum LABELS. It cannot inspect intent, but it does make the
# rule executable: any future stratum named for an ensemble or embedding signal stops
# the notebook here rather than reaching the sample. Read it together with the design
# rule it enforces -- "no stratum may be defined solely by ensemble or embedding
# output" (design 8) -- and keep the naming honest, because the assert trusts it.
stratum_definitions = set(strata["stratum"].unique())
assert not any("ensemble" in s or "embedding" in s for s in stratum_definitions), \
    "A stratum is defined by model output -- this sample must stay model-independent."
print("No stratum is defined by ensemble or embedding output.")

# WHAT THE COUNTS MEAN: one row per participant, and the row carries ONE of that
# participant's three items -- the three items are scored separately, but no reader
# sees more than one of them from the same person. Roughly up to 60 + 198 + 50,
# landing near the 250-300 target.
print("\nStratum sizes (rows; one row per participant):")
print(strata["stratum"].value_counts().to_string())

# All three items must reach the sample. The draw is stratified on item_type inside
# stratum (a), so a missing item means the pool lost a whole open-end upstream, and the
# sample would then anchor two items while every later notebook reports three. Checked,
# rather than left for the reader to spot in the counts. In smoke mode the sample is 20
# rows from 20 participants and may legitimately miss an item, so there it is printed
# and no more.
print("\nItems covered:")
print(strata["item_type"].value_counts().to_string())
_items = set(strata["item_type"])
if RUN_MODE == "full":
    assert _items == {"ranking", "backward", "forward"}, (
        f"The drawn sample covers {sorted(_items)}, not all three open-ended items. "
        "Check that openends_long.csv still carries ranking, backward and forward.")
else:
    print("  (smoke mode: too few rows to expect all three.)")

_pool_parts = gold_pool.response_id.nunique()
print(f"\nTotal rows drawn: {len(strata)} = {strata.response_id.nunique()} "
      f"participants, out of the {_pool_parts} in the pool "
      "(target roughly 250-300, MEASUREMENT_DESIGN.md design 8)")
# A count inside the target band is not on its own evidence of a healthy draw: a corpus
# reduced to 300 participants by an id mismatch also yields about 300 rows. The pool a
# full run draws from is the whole analytic sample, so the sample should be a small
# fraction of it; taking half of it means the "random slice" is close to a census.
if RUN_MODE == "full" and len(strata) > 0.5 * _pool_parts:
    print(f"  !! WARNING: {len(strata)} rows out of {_pool_parts} pool participants is "
          "more than half the pool.")
    print("     Expect roughly 300 out of about 948. A pool this small means the corpus")
    print("     or the analytic restriction lost most of the participants -- read the")
    print("     pool and restriction lines above before using either file written here.")

No stratum is defined by ensemble or embedding output.

Stratum sizes (rows; one row per participant):
stratum
random_stratified      198
survey_disagreement     60
open_readthrough        50

Items covered:
item_type
ranking     105
forward     102
backward    101

Total rows drawn: 308 = 308 participants, out of the 948 in the pool (target roughly 250-300, MEASUREMENT_DESIGN.md design 8)


In [17]:
# === FREEZE THE SEED/TEST SPLIT, ONCE ===
# CONSUMES: strata.   PRODUCES: split_assign.
#
# WHY THE SPLIT IS FROZEN BEFORE ANY SCORING HAPPENS (design 8, anti-circularity
# rule 4). The drawn sample has three colliding jobs: it is the validity criterion, it
# supplies the embedding reference passages, and it supplies the dictionary lexicon
# expansion. Without a partition, the same responses that define "correct" would be
# baked into two of the convergence streams, and part of the reported agreement would
# be trivially true -- an embedding pool built out of a response cannot fail to be
# similar to that response. One partition, made here, governs all three uses:
#   SEED -> the embedding reference pools (notebook 04 reads this file to build them),
#           the rubric's held-out anchors (notebook 05 draws its wording from the SEED
#           half only), and any lexicon expansion, which is a hand edit to notebook 01.
#   TEST -> EVERY gate and convergence number computed against this sample.
# The split needs nothing from a coder and nothing from a model. It is a seeded
# shuffle of rows that already exist, which is why it is frozen here rather than left
# waiting on anybody.
#
# Stratify the split by stratum so both halves span the strata. Stratifying matters:
# the survey-disagreement and open-read-through strata are where the hard and
# mixed-motive cases live, and a gate has to be evaluated on those cases, not only on
# the easy random slice. An unstratified coin flip could leave the TEST half short of
# exactly the rows that make the gate a test.
gc = strata.copy()
# A positional key, one integer per row of `strata`. Membership is marked on the row
# position because the shuffle-and-halve below works on positions; the id pair would do
# as well here, since the assembly cell has just proved one row per participant.
gc["_r"] = np.arange(len(gc))
seed_ids, test_ids = [], []
for strat, g in gc.groupby("stratum"):
    g = g.sample(frac=1.0, random_state=RANDOM_SEED)   # shuffle within stratum
    # Integer halving: an odd stratum sends the extra row to TEST, which is the right
    # direction -- TEST carries every gate number.
    half = len(g) // 2
    seed_ids.extend(g.iloc[:half]["_r"].tolist())
    test_ids.extend(g.iloc[half:]["_r"].tolist())
gc["gold_split"] = np.where(gc["_r"].isin(seed_ids), "SEED", "TEST")
# Every row has to have been PLACED by the loop, not landed on "TEST" by falling
# through np.where. groupby("stratum") silently drops rows whose stratum is missing, and
# such a row would be labelled TEST without ever having been shuffled into it -- then
# counted in every gate number computed on the TEST half.
assert len(seed_ids) + len(test_ids) == len(gc), (
    f"{len(gc) - len(seed_ids) - len(test_ids)} row(s) were never assigned to SEED or "
    "TEST. The usual cause is a row with a missing `stratum`, which groupby drops.")
split_assign = gc[["response_id", "item_type", "stratum", "gold_split"]].copy()

print(f"Split frozen from RANDOM_SEED = {RANDOM_SEED}.")
print(split_assign["gold_split"].value_counts().to_string())
# Roughly equal halves, and both should contain every stratum. A stratum missing from
# TEST means the gate for that region cannot be evaluated at all.
print("\nRows per stratum in each half:")
print(pd.crosstab(split_assign["stratum"], split_assign["gold_split"]).to_string())

Split frozen from RANDOM_SEED = 20260629.
gold_split
TEST    154
SEED    154

Rows per stratum in each half:
gold_split           SEED  TEST
stratum                        
open_readthrough       25    25
random_stratified      99    99
survey_disagreement    30    30


In [18]:
# === THE DRAWN SAMPLE DOUBLES AS THE CODER TEMPLATE ===
# CONSUMES: strata, FACET_NAMES.   PRODUCES: template.
#
# strata_sample.csv is written with blank facet columns and a blank holistic column,
# so the optional human arm (HUMAN_GOLD = True) can be switched on later without
# re-running anything here. Under the registration nobody has to fill them in: prereg
# 6.7 puts an ensemble of language models in place of human raters and prescribes no
# human-coding step. The blank columns cost one line each and keep that door open.
#
# WHY THE COLUMNS ARE SO FEW. Only four columns of context travel with the sample: the
# id pair, the verbatim text and the stratum. No survey scores, no dictionary counts,
# no model output -- a coder is blind to all of it (design 8), which is the whole
# basis for treating human codes as an independent criterion. Anything downstream that
# needs the dictionary counts joins them back on (response_id, item_type).
#
# Both asserts are cheap, and a sample written with blank text would waste a whole
# reading pass before anyone noticed.
assert "text" in strata.columns, \
    "strata carries no text column -- the sample would be written as blank rows."
assert (strata["text"].astype(str).str.strip() != "").all(), \
    "a drawn row has empty text -- the pool is drawn from non-blank responses only."
template = strata[["response_id", "item_type", "text", "stratum"]].copy()
for f in FACET_NAMES:
    template[f] = ""          # blank: 0 absent / 1 clearly present, or 0-2 intensity
# Multi-label, never forced to one winner: several facets may be 1 on one response,
# which is the property E4 exists to measure.
template["holistic"] = ""     # principled | vengeful | both | neither
template["coder_id"] = ""     # filled in per coder file
# `notes` is where a reader records a boundary call they disagree with -- the raw
# material for the divergence log.
template["notes"] = ""
print(f"Sample assembled: {len(template)} rows, {len(template.columns)} columns "
      f"({len(FACET_NAMES)} of them blank facet columns).")

Sample assembled: 308 rows, 16 columns (9 of them blank facet columns).


## Save and check

In [19]:
# === BEFORE THE WRITE: IS A POSTED SPLIT ALREADY IN THIS FOLDER? ===
# CONSUMES: template, split_assign.   PRODUCES: OVERWRITE_OK. Writes nothing.
#
# WHY THIS IS ITS OWN CELL. A Colab cell runs as one unit, so a warning printed in the
# same cell as the write cannot be acted on: by the time a reader has read "restore it
# before going on", the file it referred to is gone. The comparison therefore happens
# here, one cell earlier, and the write cell below refuses to run without the flag this
# cell sets.
#
# WHY IT COMPARES CONTENT. A notice on every re-run is a notice nobody reads. Re-running
# this notebook unchanged rewrites the same bytes and is harmless, and that case says so
# in one line. Only a file that would actually CHANGE stops the write.
STRATA_OUT = "strata_sample.csv"
SPLIT_OUT  = "gold_split_assignment.csv"

OVERWRITE_OK = True
for _name, _frame in [(STRATA_OUT, template), (SPLIT_OUT, split_assign)]:
    _path = os.path.join(WORKDIR, _name)
    if not os.path.exists(_path):
        continue
    with open(_path) as _fh:
        _same = _fh.read() == _frame.to_csv(index=False)
    if _same:
        print(f"  [ok] {_name} is already here and is byte-for-byte what this run "
              "would write.")
        continue
    OVERWRITE_OK = False
    _rec = _load_manifest()["files"].get(_name, {})
    print("  " + "!" * 68)
    print(f"  !! {_name} IS ALREADY IN THIS FOLDER AND WOULD CHANGE.")
    print(f"  !! The one here was written by {_rec.get('written_by', 'an unknown run')} "
          f"at {_rec.get('written_at', 'an unknown time')},")
    print(f"  !! run_mode {_rec.get('run_mode', '?')}, sha {_rec.get('sha16', '?')}.")
    print("  !! If it has been posted to OSF, that posted file is the authority and")
    print("  !! this run must not replace it. NOTHING HAS BEEN OVERWRITTEN.")
    print("  !! To keep it:    leave it alone; the sample this run drew is discarded.")
    print("  !! To replace it: move or delete it, then run this cell and the write")
    print("  !!                cell below it again.")
    print("  " + "!" * 68)
if OVERWRITE_OK:
    print("  Clear to write.")

  Clear to write.


In [20]:
# === SAVE ===
# CONSUMES: template, split_assign, OVERWRITE_OK, SURVEY_ROUTE, STRATUM_B_ROUTE.
# PRODUCES: strata_sample.csv and gold_split_assignment.csv, both through done().
#
# HAZARD: gold_split_assignment.csv is the frozen artifact. Every later notebook READS
# it rather than recomputing a split, so the partition is decided exactly once, here.
# Re-running this notebook with a different RANDOM_SEED re-draws it. Once the
# assignment has been posted, that posted file is the authority, not this code.
#
# THE CONTRACT ON WHAT GOES OUT, checked on the two frames actually being written.
# Notebooks 04, 09, 12 and 99 all merge on (response_id, item_type) and none of them
# re-checks it; a duplicated row leaving here would fan out their merges and inflate
# every count they print, with no error anywhere. The assembly cell checked `strata`,
# but that was three cells ago and cells get re-run one at a time, so what matters is
# the frame in memory now.
for _name, _frame in [(STRATA_OUT, template), (SPLIT_OUT, split_assign)]:
    _dups = int(_frame.duplicated(["response_id", "item_type"]).sum())
    assert _dups == 0, (
        f"{_name} would go out with {_dups} duplicate (response_id, item_type) rows. "
        "Every notebook downstream keys on that pair and would silently fan out.")

# The route is recorded in the manifest, not only in this run log, because the log
# scrolls away and the manifest is what a reader has months later when they are asking
# what these two files actually are.
_route_note = f"retribution_scored.csv {SURVEY_ROUTE}; stratum (b) {STRATUM_B_ROUTE}"

if not OVERWRITE_OK:
    print("  [not run] Nothing written. The cell above found a file that would change;")
    print("            move or delete it, then run that cell and this one again.")
else:
    # HAZARD: a returned coder file must come back with response_id and item_type
    # untouched. Every later alignment keys on that pair; a re-sorted or re-typed id
    # column shrinks the aligned index silently rather than raising.
    template.to_csv(STRATA_OUT, index=False)
    done(STRATA_OUT, rows=len(template),
         note=("drawn sample; doubles as the coder template (blank facet columns); "
               + _route_note))
    split_assign.to_csv(SPLIT_OUT, index=False)
    done(SPLIT_OUT, rows=len(split_assign),
         note=(f"frozen from RANDOM_SEED={RANDOM_SEED}, stratified by stratum; "
               + _route_note))

if RUN_MODE == "smoke":
    print("\n  SMOKE RUN: these two files describe 20 participants, not the corpus.")
    print("  Re-run with RUN_MODE = 'full' before either file is used or posted.")

  [wrote] strata_sample.csv  (308 rows, 68,520 bytes)
  [wrote] gold_split_assignment.csv  (308 rows, 15,304 bytes)


In [21]:
# === WHAT THE TWO FILES ARE FOR, AND WHAT RUNS NEXT ===
print("SEED half -> the embedding reference pools and any lexicon expansion.")
print("TEST half -> every convergence number computed against this sample.")
print("Never report a convergence number for a stream on rows that seeded it.")
print()
# THE ORDERING RULE, from MEASUREMENT_DESIGN.md design 8: "the split assignment goes to
# OSF before the ensemble is coded so it cannot be tuned." It is actionable rather than
# ceremonial, because the notebook that scores the corpus is 06, three notebooks away:
# post the file first, then run 06.
print("ORDER: gold_split_assignment.csv goes to OSF, as part of the dated amendment,")
print("BEFORE notebook 06 scores the corpus. Posting it first is what makes it")
print("impossible to tune the split once the results are visible. Notebooks 04 and 05")
print("run in between and read the SEED half only.")

# The third of the three places the route is stated, so it is in front of whoever reads
# only the last cell of the run.
print()
print(f"Route taken: retribution_scored.csv {SURVEY_ROUTE}; stratum (b) drawn "
      f"as {STRATUM_B_ROUTE}; stratum (a) stratified on the {STRATUM_A_SOURCE}.")
if STRATUM_B_ROUTE != "survey_disagreement":
    print("  This is NOT the registered stratum (b). See the banner above.")

# The registered gate has TWO clauses and both run without a human: (B) the models
# agree at an intraclass correlation of at least .60 -- a measure of how much
# independent raters agree -- and (C) each facet correlates at least .30 with its
# matching survey scale. Clause A, agreement with human coders, is OUR addition and a
# deviation from the registration, so it is reported honestly either way rather than
# left blank.
print()
if HUMAN_GOLD:
    print("HUMAN_GOLD is True: the optional human arm reads strata_sample.csv and")
    print("reports gate clause A (agreement with human coders). That arm is a")
    print("DEVIATION from the registration and needs the OSF addendum posted first.")
else:
    print("HUMAN_GOLD is False: gate clause A (agreement with human coders) is NOT")
    print("RUN, and downstream tables say 'not run' rather than leaving a blank that")
    print("could be misread as a failure. The two registered clauses are unaffected:")
    print("B (inter-model ICC >= .60) and C (r >= .30 with the matching survey scale)")
    print("run as normal on the full corpus.")

print()
print("Next: notebook 04 builds the embedding reference pools from the SEED half.")

SEED half -> the embedding reference pools and any lexicon expansion.
TEST half -> every convergence number computed against this sample.
Never report a convergence number for a stream on rows that seeded it.

ORDER: gold_split_assignment.csv goes to OSF, as part of the dated amendment,
BEFORE notebook 06 scores the corpus. Posting it first is what makes it
impossible to tune the split once the results are visible. Notebooks 04 and 05
run in between and read the SEED half only.

Route taken: retribution_scored.csv used; stratum (b) drawn as survey_disagreement; stratum (a) stratified on the survey `deserved` composite.

HUMAN_GOLD is False: gate clause A (agreement with human coders) is NOT
RUN, and downstream tables say 'not run' rather than leaving a blank that
could be misread as a failure. The two registered clauses are unaffected:
B (inter-model ICC >= .60) and C (r >= .30 with the matching survey scale)
run as normal on the full corpus.

Next: notebook 04 builds the embedding ref